In [ ]:
# Direct NPBoost exploration configuration

# Data selection
DATASET_TYPE = "synthetic"        # "synthetic" or "real"
DATASET_NAME = "gp_gaussian"      # synthetic examples: "gp_gaussian", "brownian_gaussian"; real examples: "bike", "cars", "cows", "spotify"
FIXED_EFFECT_OPTION = "steps_1D"  # used only when DATASET_TYPE == "synthetic"
PREDICTION_SCENARIO = "in_context"  # "in_context" or "few_shot"
SPLIT_DATA_SEED = 0

# NPBoost constructor arguments. Names intentionally match src.models.npboost.NPBoost.__init__.
name = "npboost"
experiment_seed = 42
validation_metric = "rmse"  # "rmse", "crps"
device = "cpu"              # "cpu" or "cuda"
crps_samples = 20 # 20 is the default used in our experiments
max_boosting_rounds = 10     # increase for better fits; keep small for quick exploration
fixed_effect_warm_start_rounds = 0
min_delta = 0.00001
patience = 5
restore_best_model = True
compute_aux_validation_metrics = True # 
plot_intermediate = False

# Set to true if you want to use attention
use_attention = False # keep False to fit NPBoost, set to True to fit ANPBoost

# Neural Process parameters passed as np_params.
np_params = {
    "y_dim": 1,
    "r_dim": 128,
    "z_dim": 128,
    "encoder_hidden_dims": [256, 256],
    "decoder_hidden_dims": [128, 128, 128, 128],
    "n_z_samples_train": 20,
    "n_z_samples_test": 20,
    "activation": "relu",
    "dropout": 0.0,
    "use_attention": use_attention,
    "num_heads": 8, # not used if use_attention is False
}

# LightGBM fixed-effect learner parameters passed as lgbm_params.
lgbm_params = {
    "learning_rate": 0.05,
    "feature_fraction": 1.0,
    "lambda_l2": 0,
    "min_data_in_leaf": 20,
    "num_leaves": 5,
    "max_bin": 255,
    "max_depth": -1,
    "line_search_step_length": False,
    "objective": "regression",
    "verbose": -1,
    "seed": experiment_seed,
    "deterministic": True,
    "num_threads": 1,
}

# NP training parameters passed as np_train_params.
np_train_params = {
    "batch_size": 16,
    "learning_rate": 1e-3,
    "epochs_per_round": 5,
    "use_linear_lr_scheduler": False,
}

# Plotting
N_PLOT_GROUPS = 8
PLOT_SEED = 2
PLOT_DPI = 300
SAVE_PLOTS = False
PLOT_OUTPUT_DIR = "plots/notebook_npboost_direct"


# Direct NPBoost model exploration

This notebook fits `src.models.npboost.NPBoost` directly and keeps the fitted object in memory as `npboost_model`.

The selected data must already have been generated. Use `python -m scripts.data.generate_data ...` first.
You can also use our provided bash scripts, for example `scripts/bash_scripts/synthetic_experiments/gp_gaussian/data_generation.sh` for synthetic data or `scripts/bash_scripts/real_experiments/bike/generate_split_data.sh` for real data.


In [ ]:
import logging
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from hydra.utils import instantiate
from IPython.display import display
from omegaconf import OmegaConf

from src.constants import FEW_SHOT_TASK_NAME, IN_CONTEXT_TASK_NAME, TEST_SPLIT_NAME, VALIDATION_SPLIT_NAME
from src.data.data_layout import PROJECT_ROOT
from src.data.datasets import BundleFactory, ModelDataAdapter
from src.evaluation.crps import crps_eval
from src.evaluation.squared_error import point_pred_eval
from src.models.npboost import NPBoost
from src.plot.model_diagnostics import (
    plot_fixed_effect,
    plot_fixed_effect_density,
    plot_fixed_effect_history,
    plot_fixed_effect_vs_response,
    plot_pred_vs_true_scatter,
    plot_residuals_vs_predicted_single,
    plot_single_model_coverage,
    plot_single_model_predictions,
    plot_variance_components,
)
from src.utils.helpers import set_seed
from src.utils.hydra_config_helpers import load_hydra_config

logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(name)s:%(message)s")
os.chdir(PROJECT_ROOT)


In [ ]:
def normalize_dataset_type(dataset_type):
    dataset_type = dataset_type.lower()
    if dataset_type not in {"synthetic", "real"}:
        raise ValueError("DATASET_TYPE must be 'synthetic' or 'real'.")
    return dataset_type


def normalize_task_name(task_name):
    task_name = task_name.replace("-", "_")
    if task_name not in {IN_CONTEXT_TASK_NAME, FEW_SHOT_TASK_NAME}:
        raise ValueError("PREDICTION_SCENARIO must be 'in_context' or 'few_shot'.")
    return task_name


def build_data_overrides():
    dataset_type = normalize_dataset_type(DATASET_TYPE)
    task_name = normalize_task_name(PREDICTION_SCENARIO)
    overrides = [
        f"data={dataset_type}/{DATASET_NAME}",
        f"task={task_name}",
        f"data.experiment.split_data_seed={SPLIT_DATA_SEED}",
        f"data.experiment.seed={experiment_seed}",
        "wandb.enabled=false",
    ]
    if dataset_type == "synthetic":
        overrides.insert(1, f"data/synthetic/fixed_effect={FIXED_EFFECT_OPTION}")
    return overrides


def fetch_split_df(cfg, task):
    data_provider = instantiate(cfg.data.provider)
    split_data_seed = cfg.data.experiment.split_data_seed
    if cfg.data.type == "synthetic":
        return data_provider.fetch_split_df(task=task, seed=split_data_seed)
    return data_provider.fetch_split_df(
        task=task,
        response_column=cfg.data.experiment.response_name,
        seed=split_data_seed,
    )


def resolve_prediction_splits(data_bundle, task):
    if task.name == IN_CONTEXT_TASK_NAME:
        return {
            "validation_context": data_bundle.train,
            "validation_target": data_bundle.validation,
            "test_context": data_bundle.train,
            "test_target": data_bundle.test,
        }
    if task.name == FEW_SHOT_TASK_NAME:
        return {
            "validation_context": data_bundle.validation_support,
            "validation_target": data_bundle.validation_query,
            "test_context": data_bundle.test_support,
            "test_target": data_bundle.test_query,
        }
    raise ValueError(f"Unknown task: {task.name}")


def to_npboost_splits(data_bundle, splits):
    return {
        "train": ModelDataAdapter.to_npboost_data(data_bundle.train),
        "validation_context": ModelDataAdapter.to_npboost_data(splits["validation_context"]),
        "validation_target": ModelDataAdapter.to_npboost_data(splits["validation_target"]),
        "test_context": ModelDataAdapter.to_npboost_data(splits["test_context"]),
        "test_target": ModelDataAdapter.to_npboost_data(splits["test_target"]),
    }


def evaluate_npboost_predictions(predictions, target_npbd, device, crps_samples):
    _, rmse = point_pred_eval(predictions, target_npbd)
    crps = crps_eval(predictions, target_npbd, crps_samples=crps_samples, device=device)
    fixed_effect_residual_rmse = NPBoost._fixed_effect_residual_rmse_from_prediction_arrays(
        predictions,
        target_npbd,
    )
    return {
        "rmse": rmse,
        "crps": crps,
        "fixed_effect_residual_rmse": fixed_effect_residual_rmse,
    }


def combined_true_effect_lines(context_data, target_data):
    if context_data.fixed_effect_part is None or target_data.fixed_effect_part is None:
        return None, None
    true_fixed_effect = pd.concat(
        [context_data.fixed_effect_part, target_data.fixed_effect_part],
        ignore_index=True,
    )
    if context_data.random_effect_part is None or target_data.random_effect_part is None:
        return true_fixed_effect, None
    true_random_effect = pd.concat(
        [context_data.random_effect_part, target_data.random_effect_part],
        ignore_index=True,
    )
    return true_fixed_effect, true_random_effect


def save_path(filename):
    if not SAVE_PLOTS:
        return None
    output_dir = Path(PLOT_OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir / filename


In [ ]:
overrides = build_data_overrides()
cfg = load_hydra_config(overrides=overrides, disable_wandb=True)
task = instantiate(cfg.task)
split_df = fetch_split_df(cfg, task)
data_bundle = BundleFactory.create_bundle(split_df, task)
splits = resolve_prediction_splits(data_bundle, task)
npboost_splits = to_npboost_splits(data_bundle, splits)

print("Hydra data/task overrides:")
for override in overrides:
    print("  ", override)
print(f"Train rows: {len(data_bundle.train.features):,}")
print(f"Validation target rows: {len(splits['validation_target'].features):,}")
print(f"Test target rows: {len(splits['test_target'].features):,}")
print(f"Feature columns: {list(data_bundle.train.features.columns)}")


In [ ]:
if str(device).startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError("device='cuda' was requested, but CUDA is not available in this Python environment.")

set_seed(experiment_seed)
training_history = []


def log_callback(step, metrics):
    training_history.append({"step": step, **metrics})


npboost_model = NPBoost(
    name=name,
    experiment_seed=experiment_seed,
    task=task,
    validation_metric=validation_metric,
    device=device,
    crps_samples=crps_samples,
    max_boosting_rounds=max_boosting_rounds,
    min_delta=min_delta,
    patience=patience,
    restore_best_model=restore_best_model,
    np_params=OmegaConf.create(np_params),
    lgbm_params=OmegaConf.create(lgbm_params),
    np_train_params=OmegaConf.create(np_train_params),
    fixed_effect_warm_start_rounds=fixed_effect_warm_start_rounds,
    compute_aux_validation_metrics=compute_aux_validation_metrics,
    plot_intermediate=plot_intermediate,
)

fit_started = time.time()
npboost_model.fit(
    train_npbd=npboost_splits["train"],
    validation_target_npbd=npboost_splits["validation_target"],
    validation_context_npbd=npboost_splits["validation_context"],
    log_callback=log_callback,
)
fitting_time_s = time.time() - fit_started

validation_predictions = npboost_model.predict(
    target_npbd=npboost_splits["validation_target"],
    context_npbd=npboost_splits["validation_context"],
)
test_predictions = npboost_model.predict(
    target_npbd=npboost_splits["test_target"],
    context_npbd=npboost_splits["test_context"],
)

validation_metrics = evaluate_npboost_predictions(
    validation_predictions,
    npboost_splits["validation_target"],
    device=npboost_model.device,
    crps_samples=crps_samples,
)
test_metrics = evaluate_npboost_predictions(
    test_predictions,
    npboost_splits["test_target"],
    device=npboost_model.device,
    crps_samples=crps_samples,
)

training_history_df = pd.DataFrame(training_history)
summary = pd.DataFrame(
    [
        {"split": VALIDATION_SPLIT_NAME, **validation_metrics},
        {"split": TEST_SPLIT_NAME, **test_metrics},
    ]
)

print(f"Fitted direct NPBoost model in {fitting_time_s:.1f}s")
print(f"Best validation {validation_metric}: {npboost_model.val_best:.4f}")
print(f"Best boosting round: {npboost_model.val_metric_best_boosting_round}")
display(summary)


In [ ]:
# # Create some diagnostic plots that will be saved to disk if SAVE_PLOTS is True. 
# # These use the fitted `npboost_model` from above.
# validation_predictions_ordered = validation_predictions.reorder_to_original()
# test_predictions_ordered = test_predictions.reorder_to_original()

# validation_true_fixed, validation_true_random = combined_true_effect_lines(
#     splits["validation_context"],
#     splits["validation_target"],
# )

# if splits["validation_target"].features.shape[1] <= 2:
#     fixed_effect_plot = plot_fixed_effect(
#         features=splits["validation_target"].features,
#         fixed_effect_pred=validation_predictions_ordered.fixed_effect,
#         true_fixed_effect=splits["validation_target"].fixed_effect_part,
#         model_name=name,
#         save_path=save_path("npboost_fixed_effect.png"),
#         dpi=PLOT_DPI,
#     )
#     display(fixed_effect_plot)
# else:
#     print("Skipping fixed-effect function plot because the target data has more than two feature columns.")

# if splits["validation_target"].features.shape[1] == 1:
#     predictions_plot = plot_single_model_predictions(
#         model_name=name,
#         predictions=validation_predictions,
#         prediction_metrics=validation_metrics,
#         target_data=splits["validation_target"],
#         context_data=splits["validation_context"],
#         task=task,
#         n_groups=N_PLOT_GROUPS,
#         save_path=save_path("npboost_validation_predictions.png"),
#         dpi=PLOT_DPI,
#         seed=PLOT_SEED,
#         eval_samples=crps_samples,
#         is_gaussian_prediction=False,
#         true_fixed_effect=validation_true_fixed,
#         true_random_effect=validation_true_random,
#         prediction_split_name="Validation",
#     )
#     display(predictions_plot)

#     fixed_effect_history_plot = plot_fixed_effect_history(
#         npboost_model=npboost_model,
#         target_data=data_bundle.train,
#         true_fixed_effect=data_bundle.train.fixed_effect_part,
#         save_path=save_path("npboost_fixed_effect_history.png"),
#         dpi=PLOT_DPI,
#     )
#     display(fixed_effect_history_plot)
# else:
#     print("Skipping grouped prediction and fixed-effect-history plots because they are designed for one-dimensional feature data.")

# coverage_plot = plot_single_model_coverage(
#     model_name=name,
#     predictions=validation_predictions,
#     prediction_metrics=validation_metrics,
#     target_data=npboost_splits["validation_target"],
#     save_path=save_path("npboost_validation_coverage.png"),
#     dpi=PLOT_DPI,
#     eval_samples=crps_samples,
#     is_gaussian_prediction=False,
# )
# display(coverage_plot)

# residual_plot = plot_residuals_vs_predicted_single(
#     model_name=name,
#     predictions=validation_predictions,
#     prediction_metrics=validation_metrics,
#     target_data=npboost_splits["validation_target"],
#     save_path=save_path("npboost_validation_residuals.png"),
#     dpi=PLOT_DPI,
# )
# display(residual_plot)

# scatter_plot = plot_pred_vs_true_scatter(
#     y_true=npboost_splits["validation_target"].response,
#     y_pred=validation_predictions_ordered.mean,
#     model_name=name,
#     save_path=save_path("npboost_validation_pred_vs_true.png"),
#     dpi=PLOT_DPI,
# )
# display(scatter_plot)

# variance_plot = plot_variance_components(
#     model_name=name,
#     predictions=validation_predictions,
#     save_path=save_path("npboost_variance_components.png"),
#     dpi=PLOT_DPI,
# )
# display(variance_plot)

# fixed_effect_density_plot = plot_fixed_effect_density(
#     model_name=name,
#     predictions=validation_predictions,
#     save_path=save_path("npboost_fixed_effect_density.png"),
#     dpi=PLOT_DPI,
# )
# display(fixed_effect_density_plot)

# fixed_effect_vs_response_plot = plot_fixed_effect_vs_response(
#     val_preds=validation_predictions,
#     val_target=npboost_splits["validation_target"],
#     test_preds=test_predictions,
#     test_target=npboost_splits["test_target"],
#     model_name=name,
#     save_path=save_path("npboost_fixed_effect_vs_response.png"),
#     dpi=PLOT_DPI,
# )
# display(fixed_effect_vs_response_plot)